# Chest X-Ray Dataset -- Complete EDA Notebook

**Dataset:** Kaggle Chest X-Ray Images (Pneumonia)
**Classes:** NORMAL, PNEUMONIA
**Purpose:** Full exploratory data analysis -- no model training

---

## 1. Setup & Imports

In [1]:
import os, sys, hashlib, warnings, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter, ImageStat

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Paths
NB_DIR = Path(".").resolve()
if (NB_DIR / "app" / "data" / "chest_xray").exists():
    ROOT = NB_DIR
elif (NB_DIR.parent.parent / "app" / "data" / "chest_xray").exists():
    ROOT = NB_DIR.parent.parent
else:
    raise FileNotFoundError("Cannot find app/data/chest_xray")

DATA = ROOT / "app" / "data" / "chest_xray"
OUT = ROOT / "ml" / "eda" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

SPLITS = {"train": DATA / "train", "val": DATA / "val", "test": DATA / "test"}
CLS = ["NORMAL", "PNEUMONIA"]
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
PAL = {"NORMAL": "#4C72B0", "PNEUMONIA": "#DD8452"}

print(f"Dataset: {DATA}")
print(f"Output:  {OUT}")
print(f"Python:  {sys.version}")


Dataset: C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-project\hms-ai\app\data\chest_xray
Output:  C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-project\hms-ai\ml\eda\outputs
Python:  3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]


## 2. Dataset Structure Check

In [2]:
for sn, sd in SPLITS.items():
    for c in CLS:
        d = sd / c
        n = sum(1 for f in d.iterdir() if f.suffix.lower() in EXT) if d.exists() else 0
        status = "OK" if d.exists() else "MISSING"
        print(f"  {sn}/{c}: {n} images  [{status}]")


  train/NORMAL: 1341 images  [OK]
  train/PNEUMONIA: 3875 images  [OK]
  val/NORMAL: 8 images  [OK]
  val/PNEUMONIA: 8 images  [OK]
  test/NORMAL: 234 images  [OK]
  test/PNEUMONIA: 390 images  [OK]


## 3. Full Dataset Scan

In [3]:
t0 = time.time()
records = []
for sn, sd in SPLITS.items():
    for c in CLS:
        d = sd / c
        if not d.exists(): continue
        for f in sorted(d.iterdir()):
            if f.suffix.lower() not in EXT: continue
            rec = {"split": sn, "class": c, "filename": f.name, "path": str(f),
                   "file_size_kb": f.stat().st_size / 1024.0,
                   "width": 0, "height": 0, "aspect_ratio": 0.0, "mode": "",
                   "brightness": 0.0, "contrast": 0.0, "sharpness": 0.0,
                   "corrupted": False, "md5": ""}
            try:
                img = Image.open(f); img.load()
                rec["width"], rec["height"] = img.size
                rec["aspect_ratio"] = round(rec["width"] / max(rec["height"], 1), 4)
                rec["mode"] = img.mode
                gray = img.convert("L")
                st = ImageStat.Stat(gray)
                rec["brightness"] = round(st.mean[0], 2)
                rec["contrast"] = round(st.stddev[0], 2)
                edges = gray.filter(ImageFilter.Kernel(
                    size=(3,3), kernel=[0,1,0,1,-4,1,0,1,0], scale=1, offset=128))
                rec["sharpness"] = round(float(ImageStat.Stat(edges).stddev[0]), 2)
                img.close()
            except Exception:
                rec["corrupted"] = True
            try:
                h = hashlib.md5()
                with open(f, "rb") as fh:
                    for chunk in iter(lambda: fh.read(65536), b""): h.update(chunk)
                rec["md5"] = h.hexdigest()
            except Exception: pass
            records.append(rec)

df = pd.DataFrame(records)
ok = df[~df["corrupted"]]
print(f"Scanned {len(df):,} images in {time.time()-t0:.1f}s")
print(f"Corrupted: {df['corrupted'].sum()}")
df.head()


Scanned 5,856 images in 65.4s
Corrupted: 0


,split,class,filename,path,file_size_kb,width,height,aspect_ratio,mode,brightness,contrast,sharpness,corrupted,md5
0,train,NORMAL,IM-0115-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,850.375000,2090,1858,1.1249,L,128.91,62.30,8.52,False,20e1b8286b2d4e7a9869d42abd1cbf91
1,train,NORMAL,IM-0117-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,396.782227,1422,1152,1.2344,L,100.65,59.81,10.40,False,9299d3c610ad02c31c30469c9a4c2431
2,train,NORMAL,IM-0119-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,568.983398,1810,1434,1.2622,L,121.97,68.86,9.57,False,4b873c9f031149dccd93d51e092215ca
3,train,NORMAL,IM-0122-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,460.503906,1618,1279,1.2651,L,132.99,64.97,9.38,False,f510c0d0ff70e725449ecbdf93553a70
4,train,NORMAL,IM-0125-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,440.714844,1600,1125,1.4222,L,106.22,65.09,11.00,False,3c0c7a39407e411aa83b854f8f8fa8f0


## 4. Dataset Inventory & Class Distribution

In [4]:
inv = df.groupby(["split","class"]).size().unstack(fill_value=0)
inv["Total"] = inv.sum(axis=1)
inv.loc["TOTAL"] = inv.sum()
print(inv.to_string())
inv.to_csv(OUT / "dataset_inventory.csv")
print("\nSaved: dataset_inventory.csv")


class  NORMAL  PNEUMONIA  Total
split                          
test      234        390    624
train    1341       3875   5216
val         8          8     16
TOTAL    1583       4273   5856

Saved: dataset_inventory.csv


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Bar
tr = df[df["split"]=="train"]
tr_n = (tr["class"]=="NORMAL").sum(); tr_p = (tr["class"]=="PNEUMONIA").sum()
bars = axes[0].bar(CLS, [tr_n, tr_p], color=[PAL[c] for c in CLS], width=0.45, edgecolor="white")
for b, v in zip(bars, [tr_n, tr_p]):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+40, f"{v:,}", ha="center", fontweight="bold")
axes[0].set_title("Train Class Counts", fontweight="bold"); axes[0].set_ylabel("Count")
# Pie
axes[1].pie([tr_n, tr_p], labels=[f"NORMAL\n{100*tr_n/(tr_n+tr_p):.1f}%", f"PNEUMONIA\n{100*tr_p/(tr_n+tr_p):.1f}%"],
            colors=[PAL[c] for c in CLS], startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Train Proportion", fontweight="bold")
# Grouped
splits_l = list(SPLITS.keys()); x = np.arange(len(splits_l)); w = 0.35
axes[2].bar(x-w/2, [df[(df["split"]==s)&(df["class"]=="NORMAL")].shape[0] for s in splits_l], w, label="NORMAL", color=PAL["NORMAL"])
axes[2].bar(x+w/2, [df[(df["split"]==s)&(df["class"]=="PNEUMONIA")].shape[0] for s in splits_l], w, label="PNEUMONIA", color=PAL["PNEUMONIA"])
axes[2].set_xticks(x); axes[2].set_xticklabels(splits_l); axes[2].legend()
axes[2].set_title("Per-Split Counts", fontweight="bold")
plt.suptitle("Class Distribution Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout(); plt.savefig(OUT / "class_distribution.png", dpi=150); plt.show()


## 5. Class Imbalance Analysis

In [6]:
for s in SPLITS:
    sub = df[df["split"]==s]
    n = (sub["class"]=="NORMAL").sum(); p = (sub["class"]=="PNEUMONIA").sum(); t = len(sub)
    print(f"{s:6s}  NORMAL={n:>5}  PNEUMONIA={p:>5}  Ratio={p/max(n,1):.2f}:1  Pneumonia%={100*p/max(t,1):.1f}%")
tc = df[df["split"]=="train"]
n = (tc["class"]=="NORMAL").sum(); p = (tc["class"]=="PNEUMONIA").sum(); t = n + p
w0 = t / (2*max(n,1)); w1 = t / (2*max(p,1))
print(f"\nClass weights: NORMAL={w0:.4f}  PNEUMONIA={w1:.4f}")
print(f"NORMAL gets {w0/w1:.2f}x the gradient weight")


train   NORMAL= 1341  PNEUMONIA= 3875  Ratio=2.89:1  Pneumonia%=74.3%
val     NORMAL=    8  PNEUMONIA=    8  Ratio=1.00:1  Pneumonia%=50.0%
test    NORMAL=  234  PNEUMONIA=  390  Ratio=1.67:1  Pneumonia%=62.5%

Class weights: NORMAL=1.9448  PNEUMONIA=0.6730
NORMAL gets 2.89x the gradient weight


## 6. Corrupted Image Detection

In [7]:
bad = df[df["corrupted"]]
if len(bad) == 0: print("No corrupted images found.")
else:
    print(f"{len(bad)} corrupted:")
    for _, r in bad.iterrows(): print(f"  [{r['split']}/{r['class']}] {r['filename']}")


No corrupted images found.


## 7. Image Dimensions Analysis

In [8]:
for s in SPLITS:
    sub = ok[ok["split"]==s]
    if sub.empty: continue
    print(f"{s:6s}  W: {sub['width'].min():>5}-{sub['width'].max():<5} avg={sub['width'].mean():.0f}  "
          f"H: {sub['height'].min():>5}-{sub['height'].max():<5} avg={sub['height'].mean():.0f}")
print(f"\nUnique WxH: {len(ok.groupby(['width','height']))}")
print(f"Modes: {dict(ok['mode'].value_counts())}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for c in CLS:
    sub = ok[ok["class"]==c]
    axes[0,0].hist(sub["width"], bins=50, alpha=0.6, color=PAL[c], label=c)
    axes[0,1].hist(sub["height"], bins=50, alpha=0.6, color=PAL[c], label=c)
axes[0,0].set_title("Width Distribution", fontweight="bold"); axes[0,0].legend()
axes[0,1].set_title("Height Distribution", fontweight="bold"); axes[0,1].legend()
axes[1,0].hist(ok["aspect_ratio"], bins=50, color="#2ecc71", alpha=0.8)
axes[1,0].axvline(1.0, color="red", ls="--", label="Square"); axes[1,0].legend()
axes[1,0].set_title("Aspect Ratio", fontweight="bold")
for c in CLS:
    sub = ok[ok["class"]==c].sample(min(300, len(ok[ok["class"]==c])), random_state=42)
    axes[1,1].scatter(sub["width"], sub["height"], alpha=0.3, s=8, label=c, color=PAL[c])
axes[1,1].set_title("Width vs Height", fontweight="bold"); axes[1,1].legend()
plt.suptitle("Dimension Analysis", fontsize=15, fontweight="bold"); plt.tight_layout()
plt.savefig(OUT / "dimension_analysis.png", dpi=150); plt.show()


train   W:   384-2916  avg=1321  H:   127-2663  avg=968
val     W:   968-1776  avg=1348  H:   592-1416  avg=1003
test    W:   728-2752  avg=1388  H:   344-2713  avg=992

Unique WxH: 4803
Modes: {'L': np.int64(5573), 'RGB': np.int64(283)}


## 8. Aspect Ratio Analysis

In [9]:
ar = ok["aspect_ratio"]
sq = ((ar>=0.95)&(ar<=1.05)).sum(); ls = (ar>1.05).sum(); pt = (ar<0.95).sum()
print(f"Range: {ar.min():.3f} - {ar.max():.3f}  Mean: {ar.mean():.3f}")
print(f"Square: {sq} ({100*sq/len(ar):.1f}%)  Landscape: {ls} ({100*ls/len(ar):.1f}%)  Portrait: {pt} ({100*pt/len(ar):.1f}%)")
extreme = ok[(ok["aspect_ratio"]>2.5)|(ok["aspect_ratio"]<0.5)]
if len(extreme) > 0:
    print(f"\nExtreme ratios ({len(extreme)}):")
    for _, r in extreme.head(5).iterrows():
        print(f"  [{r['split']}/{r['class']}] {r['filename']} {r['width']}x{r['height']} ar={r['aspect_ratio']:.3f}")


Range: 0.835 - 3.379  Mean: 1.443
Square: 123 (2.1%)  Landscape: 5709 (97.5%)  Portrait: 24 (0.4%)

Extreme ratios (27):
  [train/PNEUMONIA] person1311_bacteria_3312.jpeg 490x189 ar=2.593
  [train/PNEUMONIA] person1642_bacteria_4352.jpeg 724x286 ar=2.531
  [train/PNEUMONIA] person1669_bacteria_4422.jpeg 482x163 ar=2.957
  [train/PNEUMONIA] person1677_bacteria_4444.jpeg 500x198 ar=2.525
  [train/PNEUMONIA] person1679_bacteria_4448.jpeg 480x167 ar=2.874


## 9. File Size Analysis

In [10]:
for c in CLS:
    sub = ok[ok["class"]==c]; s = sub["file_size_kb"]
    print(f"{c:12s}  Avg={s.mean():>7.1f} KB  Median={s.median():>7.1f} KB  Min={s.min():.1f}  Max={s.max():.1f}")
ratio = ok[ok["class"]=="NORMAL"]["file_size_kb"].mean() / max(ok[ok["class"]=="PNEUMONIA"]["file_size_kb"].mean(), 1)
print(f"\nNORMAL images are {ratio:.1f}x larger than PNEUMONIA on average")

fig, ax = plt.subplots(figsize=(10, 5))
for c in CLS:
    ax.hist(ok[ok["class"]==c]["file_size_kb"], bins=50, alpha=0.55, label=c, color=PAL[c], edgecolor="white")
ax.set_xlabel("File Size (KB)"); ax.set_title("File Size Distribution", fontweight="bold"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "file_size_distribution.png", dpi=150); plt.show()


NORMAL        Avg=  538.2 KB  Median=  506.0 KB  Min=45.1  Max=2357.8
PNEUMONIA     Avg=   83.2 KB  Median=   69.7 KB  Min=5.3  Max=583.0

NORMAL images are 6.5x larger than PNEUMONIA on average


## 10. Brightness & Contrast Analysis

In [11]:
for c in CLS:
    sub = ok[ok["class"]==c]
    print(f"{c:12s}  Brightness: mean={sub['brightness'].mean():.1f} std={sub['brightness'].std():.1f}  "
          f"Contrast: mean={sub['contrast'].mean():.1f} std={sub['contrast'].std():.1f}")
dark = ok[ok["brightness"]<30]; bright = ok[ok["brightness"]>220]
print(f"\nVery dark (<30): {len(dark)}  Very bright (>220): {len(bright)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, c in enumerate(CLS):
    sub = ok[ok["class"]==c]; color = PAL[c]
    axes[i].hist(sub["brightness"], bins=40, color=color, alpha=0.85, edgecolor="white")
    axes[i].axvline(sub["brightness"].mean(), color="black", ls="--", label=f"Mean={sub['brightness'].mean():.0f}")
    axes[i].set_title(f"{c} -- Brightness", fontweight="bold"); axes[i].legend()
plt.tight_layout(); plt.savefig(OUT / "brightness_histograms.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([ok[ok["class"]==c]["contrast"].values for c in CLS], labels=CLS, patch_artist=True)
for patch, c in zip(bp["boxes"], PAL.values()): patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_title("Contrast: NORMAL vs PNEUMONIA", fontweight="bold")
plt.tight_layout(); plt.savefig(OUT / "contrast_boxplot.png", dpi=150); plt.show()


NORMAL        Brightness: mean=122.6 std=13.5  Contrast: mean=61.3 std=5.8
PNEUMONIA     Brightness: mean=122.8 std=19.9  Contrast: mean=55.4 std=10.0

Very dark (<30): 0  Very bright (>220): 1


## 11. Pixel Intensity Histograms

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, c in enumerate(CLS):
    sub = ok[(ok["class"]==c)&(ok["split"]=="train")].sample(min(150,len(ok[ok["class"]==c])), random_state=42)
    all_px = []
    for _, row in sub.iterrows():
        try:
            img = Image.open(row["path"]).convert("L")
            all_px.append(np.array(img).ravel()); img.close()
        except: pass
    if all_px:
        merged = np.concatenate(all_px)
        axes[i].hist(merged, bins=64, color=PAL[c], alpha=0.85, edgecolor="white", density=True)
        axes[i].set_title(f"{c} -- Pixel Intensity", fontweight="bold")
        axes[i].set_xlabel("Pixel Value (0-255)")
plt.tight_layout(); plt.savefig(OUT / "pixel_intensity_histograms.png", dpi=150); plt.show()


## 12. Sharpness / Blur Estimation

In [13]:
for c in CLS:
    s = ok[ok["class"]==c]["sharpness"]
    print(f"{c:12s}  mean={s.mean():.1f}  median={s.median():.1f}  min={s.min():.1f}  max={s.max():.1f}")
p5 = ok["sharpness"].quantile(0.05)
blurry = ok[ok["sharpness"]<p5]
print(f"\n5th percentile: {p5:.1f}  Potentially blurry: {len(blurry)}")

fig, ax = plt.subplots(figsize=(10, 5))
for c in CLS:
    vals = ok[ok["class"]==c]["sharpness"]; p99 = vals.quantile(0.99)
    ax.hist(vals.clip(upper=p99), bins=40, alpha=0.55, label=c, color=PAL[c], edgecolor="white")
ax.set_title("Sharpness Distribution", fontweight="bold"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "sharpness_distribution.png", dpi=150); plt.show()


NORMAL        mean=11.0  median=10.7  min=5.3  max=34.8
PNEUMONIA     mean=9.4  median=8.9  min=4.0  max=49.3

5th percentile: 7.0  Potentially blurry: 290


## 13. Duplicate Filename Detection

In [14]:
dup_names = df.groupby("filename").filter(lambda x: len(x)>1)
if dup_names.empty: print("No duplicate filenames.")
else:
    print(f"{dup_names['filename'].nunique()} filename(s) in multiple locations:")
    for name, grp in dup_names.groupby("filename"):
        locs = ", ".join(row["split"] + "/" + row["class"] for _, row in grp.iterrows())
        print(f"  {name} -> {locs}")


No duplicate filenames.


## 14. Exact Duplicate Detection (MD5)

In [15]:
hashed = df[(df["md5"]!="")&(~df["corrupted"])]
dup_h = hashed.groupby("md5").filter(lambda x: len(x)>1)
n_groups = dup_h["md5"].nunique()
n_redundant = len(dup_h) - n_groups
if n_groups == 0: print("No exact duplicates.")
else:
    print(f"{n_groups} duplicate group(s), {n_redundant} redundant image(s):\n")
    for md5, grp in list(dup_h.groupby("md5"))[:10]:
        paths = [f"{r['split']}/{r['class']}/{r['filename']}" for _,r in grp.iterrows()]
        print(f"  [{len(grp)}x] {paths[0]}")
        for p in paths[1:]: print(f"       = {p}")


30 duplicate group(s), 32 redundant image(s):



  [2x] test/NORMAL/NORMAL2-IM-0246-0001-0001.jpeg
       = test/NORMAL/NORMAL2-IM-0246-0001-0002.jpeg
  [2x] train/PNEUMONIA/person1619_bacteria_4267.jpeg
       = train/PNEUMONIA/person1619_bacteria_4268.jpeg
  [2x] train/PNEUMONIA/person1349_bacteria_3436.jpeg
       = train/PNEUMONIA/person1349_bacteria_3437.jpeg
  [2x] train/PNEUMONIA/person1481_bacteria_3866.jpeg
       = train/PNEUMONIA/person1481_bacteria_3867.jpeg
  [2x] train/PNEUMONIA/person357_virus_734.jpeg
       = train/PNEUMONIA/person357_virus_735.jpeg
  [2x] train/PNEUMONIA/person1261_virus_2147.jpeg
       = train/PNEUMONIA/person1261_virus_2148.jpeg
  [2x] train/PNEUMONIA/person1430_bacteria_3695.jpeg
       = train/PNEUMONIA/person1430_bacteria_3696.jpeg
  [2x] train/PNEUMONIA/person162_virus_321.jpeg
       = train/PNEUMONIA/person162_virus_322.jpeg
  [2x] test/PNEUMONIA/person128_bacteria_606.jpeg
       = test/PNEUMONIA/person128_bacteria_607.jpeg
  [2x] train/PNEUMONIA/person688_virus_1281.jpeg
       = train/P

## 15. Leakage Detection (Train <-> Val <-> Test)

In [16]:
sh = {s: set(hashed[hashed["split"]==s]["md5"]) for s in SPLITS}
for s1, s2 in [("train","val"),("train","test"),("val","test")]:
    overlap = sh.get(s1,set()) & sh.get(s2,set())
    if overlap: print(f"LEAKAGE: {len(overlap)} identical image(s) in {s1} AND {s2}")
    else: print(f"{s1} <-> {s2}: clean")


train <-> val: clean
train <-> test: clean
val <-> test: clean


## 16. NORMAL vs PNEUMONIA -- Statistical Comparison

In [17]:
comp = ok.groupby("class").agg({
    "brightness": ["mean","median","std"], "contrast": ["mean","median","std"],
    "sharpness": ["mean","median"], "file_size_kb": ["mean","median"],
    "width": ["mean","median"], "height": ["mean","median"],
}).round(1)
print(comp.to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for c in CLS:
    sub = ok[ok["class"]==c]
    axes[0,0].hist(sub["brightness"], bins=30, alpha=0.55, label=c, color=PAL[c])
    axes[0,1].hist(sub["contrast"], bins=30, alpha=0.55, label=c, color=PAL[c])
    axes[1,0].hist(sub["sharpness"].clip(upper=sub["sharpness"].quantile(0.99)), bins=30, alpha=0.55, label=c, color=PAL[c])
    axes[1,1].hist(sub["file_size_kb"], bins=30, alpha=0.55, label=c, color=PAL[c])
for ax, title in zip(axes.flat, ["Brightness","Contrast","Sharpness","File Size (KB)"]):
    ax.set_title(title, fontweight="bold"); ax.legend()
plt.suptitle("NORMAL vs PNEUMONIA", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.savefig(OUT / "class_comparison.png", dpi=150); plt.show()


          brightness              contrast              sharpness        file_size_kb          width          height        
                mean median   std     mean median   std      mean median         mean median    mean  median    mean  median
class                                                                                                                       
NORMAL         122.6  122.4  13.5     61.3   61.5   5.8      11.0   10.7        538.2  506.0  1686.4  1654.0  1378.6  1323.0
PNEUMONIA      122.8  122.9  19.9     55.4   54.8  10.0       9.4    8.9         83.2   69.7  1195.1  1160.0   819.6   776.0


## 17. Grayscale Standardization Audit

In [18]:
mode_by_cls = ok.groupby(["class","mode"]).size().unstack(fill_value=0)
print(mode_by_cls.to_string())
non_gray = ok[ok["mode"]!="L"]
if len(non_gray) == 0: print("\nAll images are grayscale.")
else: print(f"\n{len(non_gray)} non-grayscale image(s). Use .convert('RGB') for DenseNet121.")


mode          L  RGB
class               
NORMAL     1583    0
PNEUMONIA  3990  283

283 non-grayscale image(s). Use .convert('RGB') for DenseNet121.


## 18. Feature-Class Correlation Analysis

In [19]:
try:
    from scipy.stats import pointbiserialr
    labels = (ok["class"]=="PNEUMONIA").astype(int).values
    features = ["width","height","aspect_ratio","brightness","contrast","sharpness","file_size_kb"]
    print("Point-biserial correlation with class (0=NORMAL, 1=PNEUMONIA):\n")
    for feat in features:
        r, p = pointbiserialr(labels, ok[feat].values)
        sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else ""))
        print(f"  {feat:<16s}  r={r:+.4f}  p={p:.2e}  {sig}")
except ImportError:
    print("scipy not installed -- skipping correlation analysis")


Point-biserial correlation with class (0=NORMAL, 1=PNEUMONIA):

  width             r=-0.6003  p=0.00e+00  ***
  height            r=-0.6477  p=0.00e+00  ***
  aspect_ratio      r=+0.4652  p=2.51e-312  ***
  brightness        r=+0.0050  p=7.00e-01  
  contrast          r=-0.2770  p=1.31e-103  ***
  sharpness         r=-0.2555  p=6.31e-88  ***
  file_size_kb      r=-0.7814  p=0.00e+00  ***


## 19. Mean X-Ray Images & Difference Heatmap

In [20]:
mean_imgs = {}
for c in CLS:
    sub = ok[(ok["class"]==c)&(ok["split"]=="train")].sample(min(200, len(ok[ok["class"]==c])), random_state=42)
    accum = np.zeros((256,256), dtype=np.float64); count = 0
    for _, row in sub.iterrows():
        try:
            arr = np.array(Image.open(row["path"]).convert("L").resize((256,256)), dtype=np.float64)
            accum += arr; count += 1
        except: pass
    mean_imgs[c] = (accum / max(count,1)).astype(np.float32)

diff = mean_imgs["PNEUMONIA"] - mean_imgs["NORMAL"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.patch.set_facecolor("#0d1117")
for ax in axes: ax.axis("off")
axes[0].imshow(mean_imgs["NORMAL"], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Mean NORMAL", color="white", fontweight="bold")
axes[1].imshow(mean_imgs["PNEUMONIA"], cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Mean PNEUMONIA", color="white", fontweight="bold")
axes[2].imshow(diff, cmap="RdBu_r", vmin=-40, vmax=40)
axes[2].set_title("Difference", color="white", fontweight="bold")
axes[3].imshow(np.abs(diff), cmap="hot", vmin=0, vmax=40)
axes[3].set_title("Abs Diff (Hotspots)", color="white", fontweight="bold")
plt.suptitle("Mean X-Ray per Class", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "mean_xray_comparison.png", dpi=150, facecolor=fig.get_facecolor()); plt.show()


## 20. Sample Image Grids

In [21]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Sample Chest X-Rays (Train)", fontsize=14)
for row, c in enumerate(CLS):
    sub = ok[(ok["class"]==c)&(ok["split"]=="train")].head(5)
    for col, (_, rec) in enumerate(sub.iterrows()):
        try:
            img = Image.open(rec["path"]).convert("L")
            axes[row,col].imshow(np.array(img), cmap="gray")
            axes[row,col].set_title(f"{c}\n{rec['width']}x{rec['height']}", fontsize=8)
        except: pass
        axes[row,col].axis("off")
    for col in range(len(sub), 5): axes[row,col].axis("off")
plt.tight_layout(); plt.savefig(OUT / "sample_grid.png", dpi=150); plt.show()


## 21. Validation Split Proposal

In [22]:
val_total = len(df[df["split"]=="val"])
train_total = len(df[df["split"]=="train"])
print(f"Current val set: {val_total} images (Kaggle original)")
if val_total < 50:
    proposed_val = int(train_total * 0.15)
    print(f"\nProposed: carve {proposed_val} images (15%) from train as stratified val")
    print(f"  New train: ~{train_total - proposed_val}  New val: ~{proposed_val}")
    print(f"  Old val: {val_total} -> New val: {proposed_val} ({proposed_val//max(val_total,1)}x larger)")
else:
    print("Val set is adequate.")


Current val set: 16 images (Kaggle original)

Proposed: carve 782 images (15%) from train as stratified val
  New train: ~4434  New val: ~782
  Old val: 16 -> New val: 782 (48x larger)


## 22. Export Metadata CSV

In [23]:
csv_cols = ["split","class","filename","width","height","aspect_ratio",
            "file_size_kb","mode","brightness","contrast","sharpness","corrupted"]
df[csv_cols].to_csv(OUT / "image_metadata.csv", index=False)
print(f"Saved: image_metadata.csv ({len(df)} rows)")


Saved: image_metadata.csv (5856 rows)


## 23. Final EDA Summary & Recommendations

In [24]:
n_total = len(df); n_corrupt = df["corrupted"].sum()
n_dups = n_redundant if "n_redundant" in dir() else 0
ratio = (ok[ok["split"]=="train"]["class"]=="PNEUMONIA").sum() / max((ok[ok["split"]=="train"]["class"]=="NORMAL").sum(), 1)
fs_n = ok[ok["class"]=="NORMAL"]["file_size_kb"].mean()
fs_p = ok[ok["class"]=="PNEUMONIA"]["file_size_kb"].mean()
n_rgb = (ok["mode"]=="RGB").sum()
n_bright = (ok["brightness"]>220).sum()

summary = f"""# Chest X-Ray EDA Summary

## Dataset
- Total images: {n_total:,}
- Corrupted: {n_corrupt}
- Train: {(df["split"]=="train").sum():,} | Val: {(df["split"]=="val").sum()} | Test: {(df["split"]=="test").sum()}

## Key Findings
- Class imbalance: {ratio:.2f}:1 (PNEUMONIA:NORMAL in train)
- Validation set: {(df["split"]=="val").sum()} images (critically small)
- Exact duplicates: {n_dups} redundant images
- Cross-split leakage: None detected
- Unique dimensions: {ok["width"].nunique()} widths
- RGB images: {n_rgb} (mixed with grayscale)
- File size gap: NORMAL avg {fs_n:.0f} KB vs PNEUMONIA {fs_p:.0f} KB ({fs_n/max(fs_p,1):.1f}x)
- Overexposed (>220): {n_bright}

## Risks
1. Severe class imbalance (2.89:1) -- use weighted loss
2. Validation set too small (16 images) -- carve 10-15% from train
3. File size disparity may cause spurious correlation
4. Mixed color modes need standardization
5. Variable dimensions require mandatory resize

## Recommendations
1. Remove duplicate images
2. Expand validation split (stratified 15% from train)
3. Use weighted CrossEntropyLoss
4. Resize(256) + CenterCrop(224)
5. Convert all to RGB
6. Use F1/AUC-ROC, not accuracy
7. Apply moderate augmentation (flip, rotation, brightness jitter)

## ML Readiness Score: 6.5 / 10
"""
with open(OUT / "eda_summary.md", "w", encoding="utf-8") as f: f.write(summary)
print(summary)
print(f"Saved: eda_summary.md")


# Chest X-Ray EDA Summary

## Dataset
- Total images: 5,856
- Corrupted: 0
- Train: 5,216 | Val: 16 | Test: 624

## Key Findings
- Class imbalance: 2.89:1 (PNEUMONIA:NORMAL in train)
- Validation set: 16 images (critically small)
- Exact duplicates: 32 redundant images
- Cross-split leakage: None detected
- Unique dimensions: 867 widths
- RGB images: 283 (mixed with grayscale)
- File size gap: NORMAL avg 538 KB vs PNEUMONIA 83 KB (6.5x)
- Overexposed (>220): 1

## Risks
1. Severe class imbalance (2.89:1) -- use weighted loss
2. Validation set too small (16 images) -- carve 10-15% from train
3. File size disparity may cause spurious correlation
4. Mixed color modes need standardization
5. Variable dimensions require mandatory resize

## Recommendations
1. Remove duplicate images
2. Expand validation split (stratified 15% from train)
3. Use weighted CrossEntropyLoss
4. Resize(256) + CenterCrop(224)
5. Convert all to RGB
6. Use F1/AUC-ROC, not accuracy
7. Apply moderate augmentation (flip

---
## Output Files
All saved to `ml/eda/outputs/`:
- `dataset_inventory.csv` -- counts per split/class
- `image_metadata.csv` -- per-image statistics
- `eda_summary.md` -- final summary report
- `class_distribution.png` -- 3-panel class chart
- `dimension_analysis.png` -- 4-panel dimension plots
- `file_size_distribution.png` -- per-class file sizes
- `brightness_histograms.png` -- per-class brightness
- `contrast_boxplot.png` -- NORMAL vs PNEUMONIA contrast
- `pixel_intensity_histograms.png` -- pixel density per class
- `sharpness_distribution.png` -- per-class sharpness
- `class_comparison.png` -- 4-panel feature comparison
- `mean_xray_comparison.png` -- mean images + difference heatmap
- `sample_grid.png` -- 2x5 sample X-ray grid
